# Human Action Recognition with EfficientNetB0

**Goal:** teach a neural network to look at a photo of a person and recognise which of **15 actions** they are doing
(calling, clapping, cycling, dancing, drinking, eating, fighting, hugging, laughing, listening to music,
running, sitting, sleeping, texting, using a laptop).

**Plan**
1. **Preprocessing** – load the images and labels, split the data, resize the images.
2. **Model** – EfficientNetB0 with **transfer learning**.
3. **Training** – in two phases: first train only the new top layer, then *fine-tune* the network.
4. **Evaluation** – accuracy, training curves, confusion matrix.
5. **Predictions** – show the model's guesses on new, unseen photos.

> **Before running on Kaggle:** in the right-hand panel, set **Accelerator → GPU** and turn **Internet → On**
> (needed to download the pretrained EfficientNet weights). Then click **Run All**.

# Part 1 — Preprocessing

## 1.1 Import libraries
- **pandas** – read the CSV label files as tables
- **numpy** – maths on arrays of numbers
- **matplotlib** – charts and images
- **tensorflow / keras** – build the image pipeline and the neural network
- **scikit-learn** – split the data and measure results

In [ ]:
import os, glob
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # hide TensorFlow's technical log messages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

## 1.2 Find the dataset
On Kaggle, attached datasets live under `/kaggle/input/`. This searches for the label file,
so it works whatever the folder is called.

In [ ]:
csv_matches = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)
assert csv_matches, "Dataset not found - click 'Add Input' and attach the Human Action Recognition dataset."
DATA_DIR = os.path.dirname(csv_matches[0])
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
print("Data folder:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR))

## 1.3 Load the labels
`Training_set.csv` has two columns: the image **filename** and its **label** (the action).

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "Training_set.csv"))
print("Number of labeled images:", len(df))
df.head()

## 1.4 Images per action
Checks the classes are **balanced** (similar number of images each), so the model isn't biased toward one action.

In [ ]:
counts = df["label"].value_counts().sort_index()
print("Number of classes:", len(counts))
counts.plot(kind="barh", figsize=(8, 6), title="Images per action")
plt.xlabel("Number of images")
plt.tight_layout()
plt.show()

## 1.5 Example images

In [ ]:
sample = df.sample(12, random_state=1)
plt.figure(figsize=(12, 9))
for i, (_, row) in enumerate(sample.iterrows()):
    plt.subplot(3, 4, i + 1)
    plt.imshow(Image.open(os.path.join(TRAIN_IMG_DIR, row["filename"])))
    plt.title(row["label"])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 1.6 Turn labels into numbers
Neural networks work with numbers, so each action gets an ID (`calling → 0`, `clapping → 1`, …).

In [ ]:
class_names = sorted(df["label"].unique())
label_to_id = {name: i for i, name in enumerate(class_names)}
df["label_id"] = df["label"].map(label_to_id)
NUM_CLASSES = len(class_names)
for name, i in label_to_id.items():
    print(i, name)

## 1.7 Split into training and validation sets
- **Training set (80%)** – the model learns from these.
- **Validation set (20%)** – never trained on; used to check how well the model does on *new* photos.

The provided `Testing_set.csv` has **no labels**, so we can't measure accuracy on it — we use it at the end
only to show example predictions.

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label_id"], random_state=42)
print("Training images:  ", len(train_df))
print("Validation images:", len(val_df))

## 1.8 Image pipeline: load → resize → batch
For every image we:
1. **Load** the JPEG file.
2. **Resize** it to **224 × 224** pixels — the size EfficientNetB0 was designed for.
3. Group images into **batches of 32** — the network learns from 32 images at a time.

**Normalization:** pixel values stay at 0–255 here, because **EfficientNet normalizes the pixels itself**
(it has a built-in normalization layer). Scaling them to 0–1 ourselves would make it normalize twice.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)        # 3 channels = red, green, blue
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))   # same size for every image
    return img, label

def make_dataset(frame, shuffle):
    paths = [os.path.join(TRAIN_IMG_DIR, f) for f in frame["filename"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, frame["label_id"].values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=42)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)

images, labels = next(iter(train_ds))
print("Batch of images:", images.shape, " (32 images, 224 x 224 pixels, 3 colours)")
print("Batch of labels:", labels.shape)

## 1.9 Data augmentation
Random flips, rotations, zooms and contrast changes create slightly different versions of each photo during training,
so the model learns the *action* instead of memorizing exact pictures. This reduces **overfitting**.
Keras only applies these layers during training, never when evaluating or predicting.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

plt.figure(figsize=(12, 6))
for i in range(6):
    aug = augment(images[:1], training=True)[0]
    plt.subplot(2, 3, i + 1)
    plt.imshow(np.clip(aug.numpy(), 0, 255).astype("uint8"))
    plt.title(class_names[int(labels[0])])
    plt.axis("off")
plt.suptitle("Same photo, different random augmentations")
plt.tight_layout()
plt.show()

# Part 2 — The Model: EfficientNetB0 + Transfer Learning

**Transfer learning** = start from a network that has already learned to see.
EfficientNetB0 was trained on **ImageNet** (1.4 million photos, 1,000 categories), so it already recognises
edges, shapes, people, bikes, phones and laptops. We reuse that knowledge and only teach it our 15 actions.

**Why EfficientNetB0?** It gives very good accuracy for its size: about 4 million parameters, much smaller
than ResNet50 (25 million), so it trains quickly on Kaggle's GPU.

**Model structure**
```
Input photo (224×224×3)
   → Data augmentation          (training only)
   → EfficientNetB0 base        (pretrained, frozen at first)
   → Global average pooling     (summarise the image into 1,280 numbers)
   → Dropout 30%                (randomly switch off neurons → less overfitting)
   → Dense layer, 15 outputs    (softmax: one probability per action)
```

In [ ]:
base_model = keras.applications.EfficientNetB0(
    include_top=False,           # drop ImageNet's 1,000-class output layer
    weights="imagenet",          # load the pretrained knowledge
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base_model.trainable = False     # Phase 1: freeze the pretrained layers

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = base_model(x, training=False)   # keep the base in inference mode (important for BatchNorm)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs, name="HAR_EfficientNetB0")

model.summary()

# Part 3 — Training

### Key terms
- **Epoch** – one full pass through all training images.
- **Loss** – how wrong the model is (lower is better). We use *sparse categorical cross-entropy*, the standard loss for picking one class out of many.
- **Optimizer (Adam)** – the algorithm that adjusts the weights to reduce the loss.
- **Learning rate** – how big each adjustment step is.
- **Early stopping** – stop automatically when validation accuracy stops improving, and keep the best version.

## 3.1 Phase 1 — Train only the new top layer
The pretrained base is frozen; only the final Dense layer learns. This is fast and safe.

In [ ]:
EPOCHS_PHASE1 = 10

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, verbose=1),
]

history1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE1, callbacks=callbacks)

## 3.2 Phase 2 — Fine-tuning
Now we **unfreeze the top layers** of EfficientNetB0 and train them gently with a **very small learning rate**,
so the pretrained features adapt to human actions without destroying what was already learned.

BatchNormalization layers stay frozen — retraining them on a small dataset usually hurts accuracy.

In [ ]:
EPOCHS_PHASE2 = 10
FINE_TUNE_LAYERS = 40   # how many of the base model's last layers to unfreeze

base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Re-compile so the change takes effect, with a 100x smaller learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
print(f"Trainable parameters now: {trainable:,}")

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=len(history1.history["loss"]) + EPOCHS_PHASE2,
    initial_epoch=len(history1.history["loss"]),
    callbacks=callbacks,
)

# Part 4 — Evaluation

## 4.1 Training curves
- **Accuracy** should go up and **loss** should go down.
- If training accuracy keeps rising but validation accuracy falls, the model is **overfitting**.
- The dashed line marks where fine-tuning started.

In [ ]:
def combine(key):
    return history1.history[key] + history2.history[key]

acc, val_acc = combine("accuracy"), combine("val_accuracy")
loss, val_loss = combine("loss"), combine("val_loss")
epochs_range = range(1, len(acc) + 1)
fine_tune_start = len(history1.history["loss"]) + 0.5

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, "o-", label="Training")
plt.plot(epochs_range, val_acc, "o-", label="Validation")
plt.axvline(fine_tune_start, color="gray", linestyle="--", label="Start fine-tuning")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, "o-", label="Training")
plt.plot(epochs_range, val_loss, "o-", label="Validation")
plt.axvline(fine_tune_start, color="gray", linestyle="--", label="Start fine-tuning")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4.2 Final accuracy on the validation set

In [ ]:
val_loss_final, val_acc_final = model.evaluate(val_ds, verbose=0)
print(f"Validation accuracy: {val_acc_final:.1%}")
print(f"Validation loss:     {val_loss_final:.3f}")
print(f"(Random guessing would score {1/NUM_CLASSES:.1%})")

## 4.3 Results per action
- **Precision** – when the model says "running", how often is it right?
- **Recall** – of all the real "running" photos, how many did it find?
- **F1-score** – a balance of precision and recall.

In [ ]:
y_true = val_df["label_id"].values
y_prob = model.predict(val_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

## 4.4 Confusion matrix
Rows are the **true** action, columns are what the model **predicted**.
A strong diagonal means mostly correct predictions; bright squares off the diagonal show which actions get mixed up.

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=60, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xlabel("Predicted action"); ax.set_ylabel("True action")
ax.set_title("Confusion matrix (validation set)")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

# Most common mistakes
mistakes = [(cm[i, j], class_names[i], class_names[j])
            for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j]
print("Most common mix-ups:")
for n, true_name, pred_name in sorted(mistakes, reverse=True)[:5]:
    print(f"  {true_name:>20} predicted as {pred_name:<20} {n} times")

# Part 5 — Predictions on New Photos
Photos from the **test** folder — the model has never seen them and we don't know their labels.
The title shows the model's guess and how confident it is.

In [ ]:
test_df = pd.read_csv(os.path.join(DATA_DIR, "Testing_set.csv"))
demo = test_df.sample(12, random_state=7)

plt.figure(figsize=(14, 10))
for i, fname in enumerate(demo["filename"]):
    img, _ = load_image(os.path.join(TEST_IMG_DIR, fname), 0)
    probs = model.predict(tf.expand_dims(img, 0), verbose=0)[0]
    top = probs.argmax()
    plt.subplot(3, 4, i + 1)
    plt.imshow(Image.open(os.path.join(TEST_IMG_DIR, fname)))
    plt.title(f"{class_names[top]}\n{probs[top]:.0%} confident")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5.1 Save the model and all test predictions
Saved to `/kaggle/working/`, so they appear in the **Output** section of the notebook and can be downloaded.

In [ ]:
model.save("/kaggle/working/har_efficientnetb0.keras")

test_paths = [os.path.join(TEST_IMG_DIR, f) for f in test_df["filename"]]
test_ds = (tf.data.Dataset.from_tensor_slices((test_paths, np.zeros(len(test_paths), dtype=np.int64)))
           .map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE))
test_pred = model.predict(test_ds, verbose=0).argmax(axis=1)
test_df["label"] = [class_names[i] for i in test_pred]
test_df.to_csv("/kaggle/working/test_predictions.csv", index=False)

print("Saved model:       /kaggle/working/har_efficientnetb0.keras")
print("Saved predictions: /kaggle/working/test_predictions.csv")
test_df.head()

# Summary
| | |
|---|---|
| **Data** | 12,600 labeled photos, 15 actions, split 80% train / 20% validation |
| **Preprocessing** | resize to 224×224, label encoding, data augmentation |
| **Model** | EfficientNetB0 pretrained on ImageNet + new 15-class output layer |
| **Training** | Phase 1: train top layer (base frozen) → Phase 2: fine-tune top layers with a small learning rate |
| **Overfitting control** | augmentation, dropout, early stopping, frozen BatchNorm |
| **Result** | see validation accuracy in section 4.2 |

**Possible improvements:** a bigger model (EfficientNetB3), higher image resolution, more epochs of fine-tuning,
or more training data.